# Qwen2.5-1.5B QLoRA — Vietnamese Dialogue Rewriter (Google Colab)

Notebook chạy end-to-end trên Colab GPU **T4 16GB** (free tier đủ).

**Trước khi Run All:**
1. Menu *Runtime → Change runtime type → Hardware accelerator: T4 GPU*.
2. Sửa biến `REPO_URL` ở cell tiếp theo cho đúng repo của bạn.
3. (Tuỳ chọn) Bật `MOUNT_DRIVE = True` để lưu adapter vào Google Drive — Colab sẽ xoá `/content/` khi session kết thúc.

## 1. Lấy code

Mặc định clone từ GitHub. Nếu repo private, đổi sang HTTPS có PAT hoặc upload zip qua panel *Files* bên trái.

In [ ]:
import os, shutil, subprocess

REPO_URL = "https://github.com/<user>/qwen-lora-finetuning.git"  # sửa lại
PROJECT_DIR = "/content/qwen-lora-finetuning"

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
print("CWD:", os.getcwd())
print(os.listdir())

## 2. (Tuỳ chọn) Mount Google Drive

Để adapter sống sót sau khi runtime ngắt.

In [ ]:
MOUNT_DRIVE = False  # đổi True nếu muốn lưu adapter vào Drive
DRIVE_OUT_DIR = "/content/drive/MyDrive/qwen-dialogue-rewriter-lora"

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

## 3. Cài dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. (Tuỳ chọn) HuggingFace token

Bỏ qua nếu không cần. Để dùng: bấm icon 🔑 *Secrets* bên trái → thêm secret tên `HF_TOKEN` → bật toggle *Notebook access*.

In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("Skip HF token:", e)

## 5. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 6. Sinh dataset (~500 mẫu) và chia train/valid/test

In [ ]:
!python src/data/generate_dataset.py --target 500
!python src/data/generate_multi_turn.py --target 0 --output data/raw/dialogues_multi_turn.jsonl
!cat data/raw/dialogues.jsonl data/raw/dialogues_multi_turn.jsonl > data/raw/dialogues_merged.jsonl
!python src/data/split_data.py --input data/raw/dialogues_merged.jsonl
!ls -la data/processed/

## 7. Fine-tune QLoRA

Mặc định 3 epochs, batch 1 × grad-accum 8, LoRA r=8, NF4. Nếu OOM: sửa `configs/qwen_lora_sft.yaml` (`cutoff_len: 512`, `lora_rank: 4`, hoặc tăng `gradient_accumulation_steps`).

In [ ]:
!python src/train/train.py --config configs/qwen_lora_sft.yaml

## 8. So sánh base model vs fine-tuned trên test set

In [ ]:
!python -m src.inference.compare

In [ ]:
import json

with open("outputs/eval_results/comparison.jsonl", encoding="utf-8") as f:
    rows = [json.loads(l) for l in f]

print(f"Total: {len(rows)} samples\n")
for r in rows[:5]:
    print("DIALOGUE:", r["dialogue"])
    print("GOLD    :", r["gold"])
    print("BASE    :", r["base_output"])
    print("LORA    :", r["finetuned_output"])
    print("-" * 80)

## 9. Inference thử một hội thoại

In [ ]:
!python -m src.inference.predict \
  --conversation '[{"role":"user","content":"mở điều hoà"},{"role":"bot","content":"bạn muốn đặt bao nhiêu độ?"},{"role":"user","content":"27 độ"}]'

## 10. (Tuỳ chọn) Serve FastAPI qua ngrok

Colab không expose port public, dùng ngrok làm tunnel. Lấy auth token miễn phí tại https://dashboard.ngrok.com → thêm vào Secrets với tên `NGROK_AUTH_TOKEN`.

In [ ]:
RUN_API = False  # đổi True nếu muốn serve

if RUN_API:
    !pip install -q pyngrok
    from pyngrok import ngrok
    from google.colab import userdata
    import subprocess, time

    ngrok.set_auth_token(userdata.get("NGROK_AUTH_TOKEN"))
    api_proc = subprocess.Popen(
        ["uvicorn", "src.api.main:app", "--host", "0.0.0.0", "--port", "8000"]
    )
    time.sleep(8)
    public_url = ngrok.connect(8000)
    print("Public URL:", public_url)
    print("Test:  curl -X POST <URL>/rewrite -H 'Content-Type: application/json' -d '{...}'")

## 11. Lưu adapter

Vào Drive nếu đã mount, không thì zip lại để tải qua panel *Files*.

In [ ]:
src = "outputs/qwen-dialogue-rewriter-lora"

if MOUNT_DRIVE:
    if os.path.exists(DRIVE_OUT_DIR):
        shutil.rmtree(DRIVE_OUT_DIR)
    shutil.copytree(src, DRIVE_OUT_DIR)
    print("Copied adapter to", DRIVE_OUT_DIR)

zip_path = shutil.make_archive("/content/qwen-dialogue-rewriter-lora", "zip", src)
print("Zipped:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Auto-download skipped:", e)